In [2]:
import pandas as pd

df = pd.read_json("reviews_mercadolivre.json")
display(df)

,date,rating,content,product_url
0,09 set. 2023,5,Top.,https://produto.mercadolivre.com.br/MLB-314957...
1,19 ago. 2024,5,"Produto bom, cumpre o que promete.",https://produto.mercadolivre.com.br/MLB-314957...
2,15 fev. 2025,5,Ótima qualidade.,https://produto.mercadolivre.com.br/MLB-314957...
3,11 fev. 2025,4,Bom.,https://produto.mercadolivre.com.br/MLB-314957...
4,10 jan. 2025,5,Atendeu minhas expectativas.,https://produto.mercadolivre.com.br/MLB-314957...
...,...,...,...,...
103471,01 fev. 2023,5,Top.,https://produto.mercadolivre.com.br/MLB-830623...
103472,07 dez. 2022,5,Bom.,https://produto.mercadolivre.com.br/MLB-830623...
103473,16 mar. 2018,5,"Excelente! gel com excelente fixação, rende mu...",https://produto.mercadolivre.com.br/MLB-830623...
103474,11 mar. 2024,5,Comprei a linha inteira e simplesmente amei a ...,https://produto.mercadolivre.com.br/MLB-357687...


In [3]:
df = df.drop(["product_url", "date"], axis = 1)
df

,rating,content
0,5,Top.
1,5,"Produto bom, cumpre o que promete."
2,5,Ótima qualidade.
3,4,Bom.
4,5,Atendeu minhas expectativas.
...,...,...
103471,5,Top.
103472,5,Bom.
103473,5,"Excelente! gel com excelente fixação, rende mu..."
103474,5,Comprei a linha inteira e simplesmente amei a ...


In [4]:
df = df[["content", "rating"]]
df

,content,rating
0,Top.,5
1,"Produto bom, cumpre o que promete.",5
2,Ótima qualidade.,5
3,Bom.,4
4,Atendeu minhas expectativas.,5
...,...,...
103471,Top.,5
103472,Bom.,5
103473,"Excelente! gel com excelente fixação, rende mu...",5
103474,Comprei a linha inteira e simplesmente amei a ...,5


In [5]:
df.loc[:, "classe"] = pd.cut(
    df["rating"],
    bins = [0, 2, 3, 5],
    labels = ["ruim", "neutro", "bom"]

)
df

/tmp/ipykernel_8590/1379053221.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[:, "classe"] = pd.cut(


,content,rating,classe
0,Top.,5,bom
1,"Produto bom, cumpre o que promete.",5,bom
2,Ótima qualidade.,5,bom
3,Bom.,4,bom
4,Atendeu minhas expectativas.,5,bom
...,...,...,...
103471,Top.,5,bom
103472,Bom.,5,bom
103473,"Excelente! gel com excelente fixação, rende mu...",5,bom
103474,Comprei a linha inteira e simplesmente amei a ...,5,bom


In [6]:
df["classe"].value_counts()

,count
classe,
bom,96948
ruim,4277
neutro,2251


In [7]:
df_bom = df[df["classe"] == "bom"].sample(4000, random_state = 25)
df_neutro = df[df["classe"] == "neutro"]
df_ruim = df[df["classe"] == "ruim"]

df_balanceado = pd.concat([df_bom, df_neutro, df_ruim])
df_balanceado["classe"].value_counts()

,count
classe,
ruim,4277
bom,4000
neutro,2251


In [8]:
emoji_map = {
    # positivos
    "😀": "feliz",
    "😃": "feliz",
    "😄": "feliz",
    "😁": "feliz",
    "😊": "feliz",
    "🙂": "feliz",
    "😍": "amor",
    "🥰": "amor",
    "😘": "amor",
    "❤️": "amor",
    "💖": "amor",
    "👍": "bom",
    "👏": "bom",
    "🔥": "incrivel",
    "✨": "incrivel",

    # neutros
    "😐": "neutro",
    "😶": "neutro",
    "🤔": "duvida",
    "😑": "neutro",

    # negativos
    "😡": "raiva",
    "😠": "raiva",
    "🤬": "raiva",
    "😤": "raiva",
    "👎": "ruim",
    "💩": "ruim",
    "😞": "triste",
    "😔": "triste",
    "😢": "triste",
    "😭": "triste",
    "😩": "frustrado",
    "😫": "frustrado",
    "🥲": "frustrado"
}

def substituir_emoji(texto):
    for emoji, palavra in emoji_map.items():
        texto = texto.replace(emoji, f" {palavra} ")
    return texto

df_balanceado["content"] = df_balanceado["content"].apply(substituir_emoji)

df_balanceado

,content,rating,classe
100040,Gostei muito cheiro suave !.,4,bom
50272,"Por ser um tratamento anti- queda, requer um t...",4,bom
64176,Progressiva excelente super índico eu amei.,5,bom
67871,É a primeira vez que uso esse finalizados. Apl...,5,bom
96850,Ótimo adorei o resultado no meu cabelo bom b...,5,bom
...,...,...,...
103399,"Comprei o kit kids, mas e horrível o cabelo fi...",1,ruim
103455,Veio agua.,1,ruim
103457,"O produto muito bom , mais na descrição falava...",1,ruim
103459,Eu não senti que o produto deu o efeito espera...,1,ruim


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [10]:
import re
import nltk
from nltk.corpus import stopwords


nltk.download("stopwords")


stop_words = set(stopwords.words("portuguese"))


negacoes = {"não", "nunca", "nada", "jamais"}
stop_words = stop_words - negacoes

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [11]:
def limpar_texto(texto):
    texto = texto.lower()
    texto = re.sub(r'[^a-zà-ú\s]', '', texto)
    palavras = [p for p in texto.split() if p not in stop_words]
    return ' '.join(palavras)

df_balanceado['content_limpo']= df_balanceado['content'].apply(limpar_texto)
df_balanceado.drop(columns = ['content'], axis = 1, inplace=True)
df_balanceado

,rating,classe,content_limpo
100040,4,bom,gostei cheiro suave
50272,4,bom,tratamento anti queda requer tempo assiduidade...
64176,5,bom,progressiva excelente super índico amei
67871,5,bom,primeira vez uso finalizados aplico pontas cab...
96850,5,bom,ótimo adorei resultado cabelo bom bom
...,...,...,...
103399,1,ruim,comprei kit kids horrível cabelo fica embaraça...
103455,1,ruim,veio agua
103457,1,ruim,produto bom descrição falava ml veio ml
103459,1,ruim,não senti produto deu efeito esperado escrito ...


In [12]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('nb', MultinomialNB())
])

X = df_balanceado['content_limpo']
y = df_balanceado['classe']

In [13]:
x_treino, x_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=35)

pipeline.fit(x_treino, y_treino)

Pipeline(steps=[('tfidf', TfidfVectorizer()), ('nb', MultinomialNB())])

In [14]:
pipeline.score(x_teste, y_teste)

0.7160493827160493

In [15]:
y_pred = pipeline.predict(x_teste)
print(classification_report(y_teste, y_pred))

              precision    recall  f1-score   support

         bom       0.82      0.84      0.83       792
      neutro       0.66      0.11      0.19       449
        ruim       0.65      0.92      0.76       865

    accuracy                           0.72      2106
   macro avg       0.71      0.62      0.59      2106
weighted avg       0.72      0.72      0.66      2106



In [16]:
pipeline.predict(['Produto muito bom, gostei demais'])

array(['bom'], dtype='<U6')

In [17]:
pipeline.predict(['Nao gostei, achei de pessima qualidade'])

array(['ruim'], dtype='<U6')